<a href="https://colab.research.google.com/github/AdiY2j/CS6910_Assignment3/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
SOS_char = 0
EOS_char = 1

In [4]:
class Lang:
  def __init__(self, name):
    self.name = name
    self.word2count = {}
    self.word2index = {}
    self.index2word = {SOS_char : '<', EOS_char : '>'}
    self.n_chars = 2

  def add_word(self, word):
    for c in word:
      self.add_char(c)

  def add_char(self, char):
    if char not in self.word2index: # If char not present add it in word2index and inc counter
      self.word2index[char] = self.n_chars
      self.word2count[char] = 1
      self.index2word[self.n_chars] = char
      self.n_chars += 1
    else:
      self.word2count[char] += 1 #If char already present just increment counter

In [5]:
train_data = pd.read_csv('/content/drive/MyDrive/aksharantar_sampled/hin/hin_train.csv')
valid_data = pd.read_csv('/content/drive/MyDrive/aksharantar_sampled/hin/hin_valid.csv')

train_data = np.array(train_data)
valid_data = np.array(valid_data)

In [6]:
train_X, train_y = train_data[:,0], train_data[:,1]

In [7]:
input_lang, output_lang = Lang('eng'), Lang('hin')
for word in train_X:
  input_lang.add_word(word)
for word in train_y:
  output_lang.add_word(word)

pairs = [[train_X[i], train_y[i]] for i in range(len(train_X))]

In [8]:
print(pairs[1])
print(input_lang.n_chars, output_lang.n_chars)

['kirankant', 'किरणकांत']
28 66


In [9]:
len(output_lang.word2count)

64

In [10]:
input_word, output_word = pairs[1][0], pairs[1][1]
encoded_output = [output_lang.word2index[char] for char in output_word]
print(encoded_output)
decoded_output = [output_lang.index2word[i] for i in encoded_output]
print(decoded_output)

[9, 3, 10, 11, 9, 8, 12, 13]
['क', 'ि', 'र', 'ण', 'क', 'ा', 'ं', 'त']


In [11]:
def getIndex(lang, word):
  index = []
  for c in word:
    index.append(lang.word2index[c])

  return index

def getWordTensor(lang, word):
  index = getIndex(lang, word)
  index.append(EOS_char)
  return torch.tensor(index, dtype=torch.long, device=device).view(-1, 1)

def getTensorPairs(data):
  inputWordTensor = getWordTensor(input_lang, data[0])
  outputWordTensor = getWordTensor(output_lang, data[1])
  return (inputWordTensor, outputWordTensor)

In [12]:
class Encoder(nn.Module):
  def __init__(self, input_size, hidden_size, embedding_size, num_layers, dropout, cell_type, batch_size):
    super(Encoder, self).__init__()
    self.hidden_size = hidden_size
    self.embedding_size = embedding_size
    self.num_layers = num_layers
    self.batch_size = batch_size
    self.cell_type = cell_type
    self.embedding = nn.Embedding(input_size, embedding_size)
    self.dropout = nn.Dropout(dropout)

    match cell_type:
      case "RNN":
        self.rnn = nn.RNN(embedding_size, hidden_size, num_layers=num_layers, dropout=dropout)
      case "LSTM":
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers=num_layers, dropout=dropout)
      case "GRU":
        self.rnn = nn.GRU(embedding_size, hidden_size, num_layers=num_layers, dropout=dropout)

  def initializeHidden(self):
    return torch.zeros(self.num_layers, 1, self.hidden_size, device=device)

  def forward(self, input, hidden, cell):
    embedded = self.embedding(input).view(1, 1, -1)
    if self.cell_type == "LSTM":
      output, (hidden, cell) = self.rnn(self.dropout(embedded), (hidden, cell))
    else:
      output, hidden = self.rnn(self.dropout(embedded), hidden)
    return output, hidden, cell

In [13]:
class Decoder(nn.Module):
  def __init__(self, hidden_size, output_size, embedding_size, num_layers, dropout, cell_type, batch_size):
    super(Decoder, self).__init__()
    self.hidden_size = hidden_size
    self.output_size = output_size
    self.num_layers = num_layers
    self.batch_size = batch_size
    self.cell_type = cell_type
    self.embedding_size = embedding_size
    self.embedding = nn.Embedding(output_size, embedding_size)

    match cell_type:
      case "RNN":
        self.rnn = nn.RNN(embedding_size, hidden_size, num_layers=num_layers)
      case "LSTM":
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers=num_layers)
      case "GRU":
        self.rnn = nn.GRU(embedding_size, hidden_size, num_layers=num_layers)

    self.out = nn.Linear(hidden_size, output_size)
    self.softmax = nn.LogSoftmax(dim = 1)

  def initializeHidden(self):
    return torch.zeros(self.num_layers, 1, self.hidden_size, device=device)


  def forward(self, input, hidden, cell):
    embedded = self.embedding(input).view(1, 1, -1)
    embedded = F.relu(embedded)
    if self.cell_type == "LSTM":
      output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
    else :
      output, hidden = self.rnn(embedded, hidden)
    output = self.softmax(self.out(output[0]))
    return output, hidden, cell

In [14]:
teacher_forcing = 0.5
def train_step(encoder, decoder, inputTensor, outputTensor, enc_optimizer, dec_optimizer, criterion, max_len):
  loss = 0
  ip_len = inputTensor.size(0)
  op_len = outputTensor.size(0)
  enc_hidden = encoder.initializeHidden()
  enc_outputs = torch.zeros(max_len, encoder.hidden_size, device=device)
  enc_cell = encoder.initializeHidden()


  enc_optimizer.zero_grad()
  dec_optimizer.zero_grad()

  for i in range(ip_len):
    enc_op, enc_hidden, enc_cell = encoder(inputTensor[i], enc_hidden, enc_cell)
    enc_outputs[i] = enc_op[0, 0]


  dec_hidden = decoder.initializeHidden()
  dec_cell = enc_cell

  dec_input = torch.tensor([[SOS_char]], device=device)

  if random.random() < teacher_forcing :
    for i in range(op_len):
      dec_op, dec_hidden, dec_cell = decoder(dec_input, dec_hidden, dec_cell)
      loss += criterion(dec_op, outputTensor[i])
      dec_input = outputTensor  [i]

  else :

    for i in range(op_len):
      dec_op, dec_hidden, dec_cell = decoder(dec_input, dec_hidden, dec_cell)
      _, top_i = dec_op.topk(1)
      dec_input = top_i.squeeze().detach()
      loss += criterion(dec_op, outputTensor[i])

      if dec_input.item() == EOS_char :
        break

  loss.backward()

  enc_optimizer.step()
  dec_optimizer.step()

  return loss.item() / op_len

In [15]:
train_pairs = []
for data in pairs:
  train_pairs.append(getTensorPairs(data))

In [11]:
def custom_collate_fn(batch):
    # Unpack the batch of samples
    input_seqs, target_seqs = zip(*batch)

    # Pad sequences to the same length
    input_seqs_padded = torch.nn.utils.rnn.pad_sequence(input_seqs, batch_first=True, padding_value=0)
    target_seqs_padded = torch.nn.utils.rnn.pad_sequence(target_seqs, batch_first=True, padding_value=0)

    return input_seqs_padded, target_seqs_padded

In [18]:
def train(encoder, decoder, learning_rate, epochs):
    for epoch in range(epochs):
        print('Epoch : {}'.format(epoch+1))
        total_loss = 0
        enc_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
        dec_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)  # Fixed typo here
        loss_func = nn.NLLLoss()

        for i in tqdm(range(1, len(train_pairs) + 1)):
            loss = train_step(encoder, decoder, train_pairs[i - 1][0], train_pairs[i - 1][1], enc_optimizer, dec_optimizer, loss_func, 70)
            total_loss += loss
            if i % 1000 == 0:
                loss_avg = total_loss / 1000
                total_loss = 0
                print('Iteration : {}, Loss : {}'.format(i, loss_avg))


In [ ]:
hidden_size = 512
batch_size = 256
embedding_size = 16
dropout = 0.1
num_layers = 2
cell_type = "LSTM"


encoder = Encoder(input_lang.n_chars, hidden_size, embedding_size, num_layers, dropout, cell_type, batch_size).to(device)
decoder = Decoder(hidden_size, output_lang.n_chars, embedding_size, num_layers, dropout, cell_type, batch_size).to(device)
train(encoder, decoder, 0.001, 2)

Epoch : 1


  0%|          | 0/51199 [00:00<?, ?it/s]

Iteration : 1000, Loss : 3.013222635232346
Iteration : 2000, Loss : 2.9054929426398384
Iteration : 3000, Loss : 2.7254404002169808
Iteration : 4000, Loss : 2.5027876503863395
Iteration : 5000, Loss : 2.3325262521242505
Iteration : 6000, Loss : 2.164098208586664
Iteration : 7000, Loss : 2.0553887230864163
Iteration : 8000, Loss : 1.9092197810284803
Iteration : 9000, Loss : 1.7832220478066754
Iteration : 10000, Loss : 1.6728092356258464
Iteration : 11000, Loss : 1.602414820434733
Iteration : 12000, Loss : 1.5157150471318057
Iteration : 13000, Loss : 1.4492026266849263
Iteration : 14000, Loss : 1.4394879801068314
Iteration : 15000, Loss : 1.3892117424036117
Iteration : 16000, Loss : 1.3276432242999283
Iteration : 17000, Loss : 1.3050201978639688
Iteration : 18000, Loss : 1.3003094130380985
Iteration : 19000, Loss : 1.2636913110691652
Iteration : 20000, Loss : 1.2302450608235467
Iteration : 21000, Loss : 1.2062157494303831
Iteration : 22000, Loss : 1.1224735718036194
Iteration : 23000, Los